# Bronze Layer Ingestion

Purpose:
Read the raw Lending Club source files from OneLake and persist them as Delta tables without applying business transformations.

Bronze Principles:
- Preserve source fidelity
- No business logic
- No data cleansing
- No column removal
- Delta format for downstream processing

**Dataset:**    
Accepted Lending Club Loans

**Purpose:**    
Understand the raw data before defining cleaning and transformation rules for the Silver layer.

**Source:**     
Lending Club (2007–2018)

In [1]:
accepted_df = (
    spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv('Files/bronze/accepted_loans/accepted_2007_to_2018Q4.csv.gz')
)

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 3, Finished, Available, Finished, False)

In [2]:
accepted_df.printSchema()

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 4, Finished, Available, Finished, False)

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string 

In [3]:
display(accepted_df.limit(5))

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dd29b23b-4eea-46e6-a636-e1735ad7dab4)

In [4]:
print(f'acceped_df row count: {accepted_df.count()}')
print(f'acceped_df column count: {len(accepted_df.columns)}')


StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 6, Finished, Available, Finished, False)

acceped_df row count: 2260701
acceped_df column count: 151


In [5]:
from pyspark.sql.functions import col, count, when, lit

row_count = accepted_df.count()

null_counts = accepted_df.select([
    count(
        when(
            col(c).isNull() | (col(c) == ""),
            c
        )
    ).alias(c)
    for c in accepted_df.columns
])

null_stats = []

result = null_counts.collect()[0].asDict()

for column, null_count in result.items():
    null_percent = round((null_count / row_count) * 100, 2)
    null_stats.append((column, null_count, null_percent))

bronze_profile_df = spark.createDataFrame(
    null_stats,
    ["Column", "Null Count", "Null %"]
)

display(
    bronze_profile_df.orderBy(col("Null %").desc())
)

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 596979aa-b59c-47ba-bb95-b67d6c009d41)

# Bronze Profiling Summary

Dataset Statistics

- Rows: 2,260,701
- Columns: 151

Key Findings

- member_id is completely null.
- Hardship-related columns are over 99% null.
- Settlement-related columns are over 98% null.
- Joint applicant columns are over 95% null.
- These columns will be evaluated before the Silver transformation.

In [6]:
accepted_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_accepted_loans")

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 8, Finished, Available, Finished, False)

In [7]:
spark.read.table("bronze_accepted_loans").count()

display(
    spark.read.table("bronze_accepted_loans").limit(5)
)

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e1759e6-dc63-4a5c-9d95-90606725448a)

# Bronze Ingestion - Rejected Loans

In [8]:
rejected_df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("Files/bronze/rejected_loans/rejected_2007_to_2018Q4.csv.gz")
)

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 10, Finished, Available, Finished, False)

In [9]:
print(f"Rows    : {rejected_df.count():,}")
print(f"Columns : {len(rejected_df.columns)}")

rejected_df.printSchema()

display(rejected_df.limit(5))

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 11, Finished, Available, Finished, False)

Rows    : 27,648,741
Columns : 9
root
 |-- Amount Requested: double (nullable = true)
 |-- Application Date: date (nullable = true)
 |-- Loan Title: string (nullable = true)
 |-- Risk_Score: string (nullable = true)
 |-- Debt-To-Income Ratio: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Employment Length: string (nullable = true)
 |-- Policy Code: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 4e0cefff-aaa3-4f72-885d-55e1e508d097)

In [10]:
import re

new_columns = [
    re.sub(r'[^a-zA-Z0-9]', '_', c)
      .lower()
      .strip('_')
    for c in rejected_df.columns
]

rejected_df = rejected_df.toDF(*new_columns)

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 12, Finished, Available, Finished, False)

In [11]:
rejected_df.printSchema()

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 13, Finished, Available, Finished, False)

root
 |-- amount_requested: double (nullable = true)
 |-- application_date: date (nullable = true)
 |-- loan_title: string (nullable = true)
 |-- risk_score: string (nullable = true)
 |-- debt_to_income_ratio: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- employment_length: string (nullable = true)
 |-- policy_code: string (nullable = true)



In [12]:
rejected_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_rejected_loans")

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 14, Finished, Available, Finished, False)

In [14]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

log_df = spark.createDataFrame([
    ("pl_loan_portfolio_etl",
     "nb_load_bronze",
     "SUCCESS",
     accepted_df.count())
], ["pipeline_name", "notebook_name", "status", "rows_processed"])

log_df = log_df.withColumn("run_time", current_timestamp())

log_df.write.mode("append").saveAsTable("audit_log")

StatementMeta(, 9777365f-116f-4b62-bf96-f8e86612ec03, 16, Finished, Available, Finished, False)